# run_experiment

One experiment, end to end, from a settings file. Works on Colab and on a rented
GPU box; the only difference is which settings file you name.

Edit the first code cell and nothing else: the settings file, the branch, and the Hub repo. To run a second experiment, change
`SETTINGS_FILE` and re-run from **Run the experiment** downwards. The model stays
downloaded and the environment stays installed, so each extra run costs only its
own training time.

On a rented machine that expires, run the save cell after every experiment rather
than at the end. Everything on that box is deleted when it shuts down.

In [1]:
USE_TEST_CONFIG = False         # True: the CPU smoke settings in src/configs/test.py
SETTINGS_FILE = "configs/minimum_threshold_one_persona/rule_tests/persona_a/one_persona_minimum_threshold_cut25.json"

# Resume an interrupted run instead of starting a new one (README "Resuming a run"). Give the
# run's id, its folder under results/, and a decision step it saved a checkpoint at and finished
# evaluating. It keeps the run's id and folders, skips the work already done, finishes an
# introspection branch at that step that never did, and trains on from the next step. Only
# compute, gates, training_steps and later introspection points may differ from the run's own
# settings; anything else is refused. None starts a new run. For the run that ran out of memory:
#   SETTINGS_FILE = "configs/paper_minimum_threshold/paper_minimum_threshold_8b_16gb_resume.json"
#   RESUME_RUN_ID = "20261031-061400_paper_minimum_threshold_Qwen3-8B"
#   RESUME_FROM_STEP = 1500
RESUME_RUN_ID = None
RESUME_FROM_STEP = None

# The branch of the repo to run. On a fresh machine it is cloned; an existing clone is moved
# onto it and brought up to date, so the code is always this branch's latest commit.
BRANCH = "bootstrap-ci"

# Where finished runs are saved so they survive the machine. Must be a Hub repo you
# can write to; the owner must match the account your token belongs to.
HUB_REPO = "adamchanna/bdp-plunkett-qwen3"

# Settings files: configs/README.md maps every folder. Each file is complete (nothing inherited),
# so open it to see, or edit, every value the run uses.
#   configs/default.json                         Plunkett on Qwen3-0.6B, 4-bit, fp16 (Colab T4)
#   configs/plunkett_linear/                     atkinson, linear-4b_20gb, linear-8b_20gb
#   configs/minimum_threshold/                   the minimum threshold rule at Qwen3-8B, many personas
#   configs/paper_minimum_threshold/             the paper's design: 8b, 8b_16gb (+ _units, _resume), 0.6b, 1500_steps
#   configs/one_persona_maximum_threshold/       one persona, maximum threshold
#   configs/minimum_threshold_one_persona/       one persona, Qwen3-8B: base/ controls/ inverse/ rule_tests/ cut_sweep/,
#                                                each split into persona_a/ and persona_b/. Every experiment has a
#                                                16 GB file (5070 Ti) and a _24gb file (L4, A10, 4090)
#   configs/minimum_threshold_one_persona_qwen3_14b/   the same on Qwen3-14B: _14b_24gb, _14b_80gb
#   configs/minimum_threshold_one_persona_qwen3_32b/   the same on Qwen3-32B: _32b_48gb, _32b_80gb
#
# A card's sizing is micro_batch_size and the compute eval batch sizes, in the file itself.
#
# Introspection, set in the settings file under model_hyperparameters (README "Settings"):
#   introspection_at_steps    the decision steps stage two branches from. A list, [1500, 2500, 3000],
#                             or a range, {"start": 1000, "end": 3000, "every": 500}, which is
#                             1000, 1500, ..., 3000. null: the last decision step only
#   introspection_steps       how long each branch trains (its end point), per fold
#   introspection_batch_size  examples per introspection step
#   introspection_training    false switches stage two off altogether
#   training_steps            the last decision step; no branch point may lie past it
#   introspection_targets     what each branch trains toward, one branch each: hidden (the persona's
#                             own rule), recovered (the rule read from the model's own choices),
#                             shuffled (another persona's: format without content), random
#   introspection_remeasure_decisions  ask the held-out personas' decisions again after each branch
#   lr_schedule               null (constant) or {"type": "linear"/"cosine", "warmup_steps": W,
#                             "final_fraction": f}
#
# Which questions are asked and trained on: report_schema.questions in the settings file, each
# {"ask": true/false, "train": true/false}; every question is a folder under questions/ (README "Questions").


## Set up the machine

In [2]:
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, use USE_TEST_CONFIG = True"

Fri Nov 13 11:35:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.88                 KMD Version: 610.88        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
| 44%   49C    P5             14W /  300W |    1701MiB /  16303MiB |      5%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Clone BRANCH when the code is not already here, or move an existing clone onto it, then pin versions.
# run_me.py runs before the install: it re-pins requirements to whatever this machine
# already ships, keeping its CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

CLONE_URL = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", BRANCH, CLONE_URL], check=True)
    os.chdir("Decision_Task_Database_Experiments")
# An existing clone may sit on another branch or an older commit. This refuses rather than
# overwrite local edits: commit or stash them, or set BRANCH to the branch they are on.
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"code: branch {BRANCH}, commit {commit}")
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
# Let the CUDA allocator grow segments instead of fragmenting. Must be set before
# torch initialises CUDA, which is why it lives here and not in the GPU check.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("repo ready in", os.getcwd())

code: branch bootstrap-ci, commit 74c92b8 Reorganize one-persona configs/data into minimum_threshold_one_persona, add rule tests
repo ready in C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments


In [4]:
# %pip installs into THIS kernel's environment. A pip subprocess can land packages
# somewhere the kernel cannot import from, which surfaces later as ModuleNotFoundError.
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Scripts\python.exe -m pip install --upgrade pip


In [5]:
# Confirm the kernel can import what the pipeline needs.
#
# On JupyterHub, pip usually installs into the user site-packages, and a kernel
# that started before that folder existed does not have it on sys.path. That is
# the normal cause of ModuleNotFoundError straight after a successful install.
# Adding the folder here repairs it without a restart.
import importlib, importlib.util, os, site, sys

user_site = site.getusersitepackages()
if user_site and os.path.isdir(user_site) and user_site not in sys.path:
    sys.path.append(user_site)
    importlib.invalidate_caches()
    print("added user site-packages to sys.path:", user_site)

print("kernel python:", sys.executable)
needed = ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft", "huggingface_hub")
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("MISSING:", missing)
    print("Restart the kernel (Kernel -> Restart Kernel), run the install cell, then this cell again.")
else:
    print("all imports available")


kernel python: C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Scripts\python.exe
all imports available


In [6]:
# Confirm the GPU matches what the settings assume. bf16 needs Ampere or newer:
# the 20gb configs ask for it and the loader refuses rather than falling back.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram < 30:
        print(f"\nThis is a MIG slice of about {vram:.0f} GB, not a whole card.")
        print("Use the *_20gb configs for 4B and 8B. 14B will not fit here.")
    elif vram < 60:
        print(f"\nThis is a MIG slice of about {vram:.0f} GB. The *_20gb configs fit with room to spare.")
else:
    print("no CUDA device: only USE_TEST_CONFIG = True will run here")

torch 2.8.0+cu128
gpu NVIDIA GeForce RTX 5070 Ti
vram GB 17.1
bf16 supported True

This is a MIG slice of about 17 GB, not a whole card.
Use the *_20gb configs for 4B and 8B. 14B will not fit here.


In [7]:
# Credentials. Typed rather than written into a cell, so nothing lands in git or in
# a saved notebook. Needed only for the save cells; the public Qwen3 weights do not
# require a token.
#
# A token that the Hub rejects is discarded and you are asked again, so a typo or
# a revoked token never gets stuck in the environment.
import getpass, os
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

owner = HUB_REPO.split("/")[0]
who = None
for attempt in range(3):
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face WRITE token (blank to skip): ")
    if not token:
        break
    try:
        who = HfApi(token=token).whoami()
        os.environ["HF_TOKEN"] = token
        break
    except HfHubHTTPError as err:
        os.environ.pop("HF_TOKEN", None)
        if err.response is not None and err.response.status_code == 401:
            print("The Hub rejected that token (401). Check it is a WRITE token and not revoked. Try again.")
            continue
        raise

if who is None:
    print("no token: the run works, but saving refuses and nothing survives the machine")
else:
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Saving will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print(f"Set HUB_REPO = \"{who['name']}/bdp-plunkett-qwen3\" in the first cell and re-run it.")


C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face WRITE token (blank to skip):  ········


token belongs to: adamchanna
HUB_REPO owner 'adamchanna' matches. Saving will work.


## Run the experiment

On a rented box, go in this order and save after each one. Starting small
exercises the CUDA path, the estimator, the report parser and the results writer in
a few minutes, which is the cheapest place to find a broken setting.

    default  ->  linear-4b_20gb  ->  linear-8b_20gb

For the minimum threshold experiments, the 0.6B run first, then 8B:

    paper_minimum_threshold_0.6b  ->  paper_minimum_threshold_8b

The cell prints the plan before training starts: the rule, the model, the decision steps,
where introspection branches and for how long. Check it before walking away.

In [8]:
from src.config import load
from src.pipeline import run

from src.config import estimation_counts, eval_batch_size, introspection_points

cfg = load(None if USE_TEST_CONFIG else SETTINGS_FILE, use_test=USE_TEST_CONFIG)
mh, rule = cfg["model_hyperparameters"], cfg["decision_rule"]
points = introspection_points(cfg)
print(f"settings      {cfg['_source']}")
print(f"rule          {rule['type']}, {rule['option_count']} options, data {cfg['model_training']['data_dir']}")
print(f"model         {mh['model_name']}, {mh['quantization']}, {cfg['compute']['mixed_precision']} on {cfg['compute']['device']}")
print(f"decisions     {mh['training_steps']} steps, a checkpoint every {mh['checkpoint_every']}, "
      f"baseline {'on' if mh['baseline_before_training'] else 'off'}")
print(f"introspection " + (f"from decision steps {points}, {mh['introspection_steps']} steps per fold, "
                           f"targets {mh['introspection_targets']}"
                           + (", decisions re-measured after" if mh["introspection_remeasure_decisions"] else "")
                           if points else "off"))
print(f"learning rate {mh['learning_rate']}, schedule {mh['lr_schedule'] or 'constant'}")
questions = cfg["report_schema"].get("questions")
if questions:
    print("questions     " + ", ".join(f"{q} ({'asked, trained' if f.get('train', True) else 'asked only'})"
                                          for q, f in questions.items() if f.get("ask", True)))
print(f"estimation    {estimation_counts(cfg)} trials per persona")
print(f"eval batches  decision {eval_batch_size(cfg, 'decision')}, introspection {eval_batch_size(cfg, 'introspection')}")

RESUME = {"run_id": RESUME_RUN_ID, "from_step": RESUME_FROM_STEP} if RESUME_RUN_ID else None
print(f"resume        " + (f"{RESUME_RUN_ID} from decision step {RESUME_FROM_STEP}" if RESUME else "off: a new run"))

rows = run(cfg, resume=RESUME)

settings      configs\minimum_threshold_one_persona\rule_tests\one_persona_minimum_threshold_cut25.json <- configs\minimum_threshold_one_persona\rule_tests\..\base\one_persona_probes_a.json <- configs\minimum_threshold_one_persona\rule_tests\..\base\one_persona_no_trial_a.json <- configs\minimum_threshold_one_persona\rule_tests\..\base\one_persona_minimum_threshold.json <- configs\minimum_threshold_one_persona\rule_tests\..\base\..\..\minimum_threshold_1500_steps.json <- configs\minimum_threshold_one_persona\rule_tests\..\base\..\..\paper_minimum_threshold_8b_16gb.json <- configs\minimum_threshold_one_persona\rule_tests\..\base\..\..\default.json
rule          paper_minimum_threshold, 3 options, data data/minimum_threshold_one_persona/rule_tests/one_persona_minimum_threshold_cut25/
model         Qwen/Qwen3-8B, 4bit, bf16 on cuda
decisions     3000 steps, a checkpoint every 250, baseline on
introspection off
learning rate 0.0002, schedule constant
questions     main (asked only), screen

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\transformers\modeling_utils.py:6231: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  _ = torch.empty(byte_count // factor, dtype=torch.float16, device=device, requires_grad=False)
Loading checkpoint shards: 100%|██████████| 5/5 [00:12<00:00,  2.56s/it]


model: Qwen/Qwen3-8B | quantization=4bit | device=cuda | dtype=bfloat16
gradient checkpointing: on
finetune: lora rank 8 alpha 8 on ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']; 21,823,488 trainable parameters
base model: the untrained model, scored before decision training
evaluating base_model_step-0 on 1 personas
  screen fit (1000 trials): attribute matches the persona's own for 0/1, cut too for 0/1
  decisions: 1000 rows, accuracy 0.466, 85s
    generating: 20/50 replies, 34s
    generating: 40/50 replies, 67s
    generating: 50/50 replies, 86s
  reports [paper_minimum_threshold/main]: 50/50 parsed, 172s
    generating: 20/50 replies, 7s
    generating: 40/50 replies, 13s
    generating: 50/50 replies, 17s
  reports [paper_minimum_threshold/screen_index]: 50/50 parsed, 189s
    generating: 20/50 replies, 15s
    generating: 40/50 replies, 30s
    generating: 50/50 replies, 39s
  reports [paper_minimum_threshold/screen_cut_units]: 47/50 parsed, 228s

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  step 50/3000 loss 0.6710 acc 0.438 [mean of steps 1-50] | 136s
  step 100/3000 loss 0.4089 acc 0.518 [mean of steps 51-100] | 271s
  step 150/3000 loss 0.2460 acc 0.782 [mean of steps 101-150] | 404s
  step 200/3000 loss 0.1660 acc 0.870 [mean of steps 151-200] | 539s
  step 250/3000 loss 0.1088 acc 0.900 [mean of steps 201-250] | val loss 0.0614 val acc 0.965 weights 0.939 neither 1.000 cut_agrees 1.000 cut_overturns 0.990 (1000 trials, 73s) | largest gradient 30.75, 112 steps clipped at 1.0 | saved checkpoints\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25\decision\step-250 | 745s
evaluating decision_step-250 on 1 personas
  screen fit (1000 trials): attribute matches the persona's own for 1/1, cut too for 1/1
  decisions: 1000 rows, accuracy 0.959, 86s
    generating: 20/50 replies, 34s
    generating: 40/50 replies, 67s
    generating: 50/50 replies, 87s
  reports [paper_minimum_threshold/main]: 50/50 parsed, 172s
    generating: 20/50 replies, 6s
    generating: 40/50 re

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


... (19 outputs left out)

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  step 2800/3000 loss 0.0404 acc 0.970 [mean of steps 2751-2800] | 16901s
  step 2850/3000 loss 0.0330 acc 0.978 [mean of steps 2801-2850] | 17031s
  step 2900/3000 loss 0.0371 acc 0.970 [mean of steps 2851-2900] | 17160s
  step 2950/3000 loss 0.0321 acc 0.978 [mean of steps 2901-2950] | 17289s
  step 3000/3000 loss 0.0298 acc 0.974 [mean of steps 2951-3000] | val loss 0.0266 val acc 0.981 weights 0.965 neither 1.000 cut_agrees 1.000 cut_overturns 1.000 (1000 trials, 70s) | largest gradient 2.24, 18 steps clipped at 1.0 | saved checkpoints\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25\decision\step-3000 | 17490s
evaluating decision_step-3000 on 1 personas
  screen fit (1000 trials): attribute matches the persona's own for 1/1, cut too for 1/1
  decisions: 1000 rows, accuracy 0.980, 84s
    generating: 20/50 replies, 29s
    generating: 40/50 replies, 59s
    generating: 50/50 replies, 77s
  reports [paper_minimum_threshold/main]: 50/50 parsed, 161s
    generating: 20/50 replie

C:\Users\PunchCat\adam_experiment\Decision_Task_Database_Experiments\src\plots.py:357: DtypeWarning: Columns (19: introspection_target) have mixed types. Specify dtype option on import or set low_memory=False.
  made.append(plot_training(pd.read_csv(log_path), run_id, out_dir))


plots: results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_training.png, results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_checkpoints.png, results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_scatter_paper_minimum_threshold_main.png, results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_scatter_paper_minimum_threshold_main_no_trial.png, results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_scatter_paper_minimum_threshold_main_report_units-main.png, results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_scatter_paper_minimum_threshold_main_report_units_trial_first-main.png, results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_scatter_paper_minimum_threshold_main_threshold_met.png, results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_scatter_paper_minimum_threshold_main_threshold_not_met.png, results\plots\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25_

## Save it off the machine

Results are small and worth pushing often. Adapters are tens of megabytes, so pass
`checkpoints=False` for a quick repeat push mid-run.

In [11]:
from src.persist import push_run

push_run(rows[-1]["run_id"], HUB_REPO)

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-l

pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25
  results\plots
  checkpoints\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25


['results\\results.csv',
 'results\\train_log.csv',
 'results\\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25',
 'results\\plots',
 'checkpoints\\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25']

In [12]:
# Before you walk away: sweep everything still on disk, including any run that was
# interrupted part way. Results are appended as a run goes, so a partial run is
# still worth keeping.
from src.persist import run_ids, push_run

for rid in run_ids():
    push_run(rid, HUB_REPO)

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261029-202425_paper_minimum_threshold_Qwen3-0.6B
  results\plots


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261029-204411_paper_minimum_threshold_Qwen3-0.6B
  results\plots


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261029-211646_paper_minimum_threshold_Qwen3-0.6B
  results\plots


... (96 outputs left out)

No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified s

pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25
  results\plots
  checkpoints\20261113-113553_paper_minimum_threshold_Qwen3-8B_cut25


No files have been modified since last commit. Skipping to prevent empty commit.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


pushed to https://huggingface.co/adamchanna/bdp-plunkett-qwen3
  results\results.csv
  results\train_log.csv
  results\queue
  results\plots
